In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

In [ ]:
# pre-process the data
## drop irrelevant columns
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data

In [ ]:
## apply encoding on categorical variable for example geography, gender
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])
data

In [ ]:
# for Geography - we not directly use LableEncoder becuase if we do then it may contains value like france-1, spain-2, Germany-3 and during training it suppose 3>1 like Germany is greater then france
# - so we use OHE to encode these type of data
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geography=OneHotEncoder()
geo_encoder=one_hot_encoder_geography.fit_transform(data[["Geography"]])
geo_encoder

In [ ]:
geo_encoder.toarray()

In [ ]:
one_hot_encoder_geography.get_feature_names_out(["Geography"])

In [ ]:
geo_endcoded_df=pd.DataFrame(geo_encoder.toarray(),columns=one_hot_encoder_geography.get_feature_names_out(["Geography"]))
geo_endcoded_df

In [ ]:
## combine all the columns like one hot encoding to original data
data=pd.concat([data.drop("Geography",axis=1),geo_endcoded_df],axis=1)
data

In [ ]:
## save the encoder and scaler
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(one_hot_encoder_geography, file)

In [ ]:
## divide the dataset into dependent and independent features
X=data.drop("Exited",axis=1)
Y=data["Exited"]

## split the data into training and testing data
X_train,X_test,Y_train,Y_test=train_test_split(X,Y, test_size=0.2, random_state=42)

# scale these feature
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [ ]:
X_train

In [ ]:
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler, file)

## ANN Implementation


key parameter we are using
- 1- when we start creating ANN we have to intialize - "sequential network"
- 2- when we need to create hidden neuron we using - "Dense"
- 3- every node we use activation function - it can be - sigmoid, ReLU, Tanh (hyperbolic tangent), softmax, Leaky ReLU
- 4- "optimizers" - usefull for backword propagation - benefit in updating weight
- 5- use loss function -  based on this loss function we try to reduce gradient decendant
- 6- "matrix" - usually in classification problem our matrix is "accuracy", in regression problem we have ["mean square error", "mean absolute error"]
- Training -> Logs -> folder -> Tensorboard -> Visualization

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
## Build our ANN models
model=Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)), ## HL1 connected to input layer
    Dense(32,activation="relu"), ## HL2
    Dense(1,activation="sigmoid")
])

In [ ]:
model.summary()

In [ ]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
tf.keras.losses.BinaryCrossentropy()

In [ ]:
## compile the model
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
## Setup the tenserboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callabcks=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [ ]:
## setup Early Stopping
ealry_stopping_callback=EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True) ## patience till 5 epoch before decide gradient decendent

In [ ]:
## training the model
history=model.fit(
    X_train,Y_train,validation_data=(X_test,Y_test), epochs=100,
    callbacks=[tensorflow_callabcks, ealry_stopping_callback]
)

In [ ]:
model.save("model.h5")

In [ ]:
## load tensorboard extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
## load the pickle file